### Imports

In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.transforms import InterpolationMode
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image
import pandas as pd
import shutil
import os

### Datasets

In [ ]:
source_dirs = {
    'coriandum_sativum': 'plant_img_database/coriandum_sativum',
    'mentha_spicata': 'plant_img_database/mentha_spicata',
    'echeveria_lilicana': 'plant_img_database/echeveria_lilicana',
    'fittonia_albivenis': 'plant_img_database/fittonia_albivenis',
    'salvia_officinales': 'plant_img_database/salvia_officinales',
    'haworthia_pumila': 'plant_img_database/haworthia_pumila',
}

### Model Definition

In [ ]:
class PlantDataModule(pl.LightningDataModule):
    def __init__(self, train_dir, val_dir, test_dir, batch_size=32, num_workers=4):
        """
        Initializes the PlantDataModule with directories for training, validation, and testing datasets, along with batch size and number of workers for data loading.
        """
        super().__init__()
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.test_dir = test_dir
        self.batch_size = batch_size
        self.num_workers = num_workers

        # Use the default weights for EfficientNet_B0 to get the correct normalization values
        weights = EfficientNet_B0_Weights.DEFAULT
        self.imagenet_mean = weights.transforms().mean
        self.imagenet_std = weights.transforms().std

        # Define data augmentation and preprocessing transforms for training and evaluation
        self.train_transform = transforms.Compose([
            transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
            transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.imagenet_mean, std=self.imagenet_std),
            transforms.RandomErasing(p=0.25),
        ])

        # For validation and testing, we only apply resizing, center cropping, and normalization without augmentation
        self.eval_transform = transforms.Compose([
            transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.imagenet_mean, std=self.imagenet_std),
        ])

    def setup(self, stage=None):
        """  
        Initializes the datasets for training, validation, and testing using ImageFolder and applies the appropriate transforms.
        """
        self.train_dataset = ImageFolder(root=self.train_dir, transform=self.train_transform)
        self.val_dataset = ImageFolder(root=self.val_dir, transform=self.eval_transform)
        self.test_dataset = ImageFolder(root=self.test_dir, transform=self.eval_transform)

    def train_dataloader(self):
        """  
        Returns the DataLoader for the training dataset with shuffling enabled and appropriate settings for performance.
        """
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=(self.num_workers > 0))

    def val_dataloader(self):
        """  
        Returns the DataLoader for the validation dataset with shuffling disabled and appropriate settings for performance.
        """
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=(self.num_workers > 0))

    def test_dataloader(self):
        """
        Returns the DataLoader for the testing dataset with shuffling disabled and appropriate settings for performance.
        """
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=(self.num_workers > 0))

In [ ]:
class EfficientNetFineTuner(pl.LightningModule):
    def __init__(self, n_classes, lr_head=1e-3, lr_backbone=1e-4, weight_decay=1e-4):
        """
        A PyTorch Lightning module for fine-tuning an EfficientNet_B0 model on a custom plant classification task.
        """
        super().__init__()
        self.save_hyperparameters()

        # Load the EfficientNet_B0 model with pre-trained weights
        weights = EfficientNet_B0_Weights.DEFAULT
        self.model = efficientnet_b0(weights=weights)

        # Replace the final classification layer to match the number of classes in our dataset
        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, n_classes)

        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        """
        Forward pass through the model to get logits for the input batch of images.
        """
        return self.model(x)
    
    def freeze_backbone(self):
        """
        Freeze the backbone (feature extractor) layers of the model to prevent them from being updated during training.
        """
        # Freeze all layers in the backbone
        for param in self.model.features.parameters():
            param.requires_grad = False

        # Unfreeze the classifier head to allow training of the new layers
        for param in self.model.classifier.parameters():
            param.requires_grad = True

    def unfreeze_last_n_blocks(self, n=3):
        """
        Unfreeze the last n blocks of the backbone to allow fine-tuning of these layers.
        """
        for param in self.model.features[-n:].parameters():
            param.requires_grad = True

    def training_step(self, batch, batch_idx):
        """
        Perform a training step by computing the loss and accuracy for the current batch.
        Metrics logged for monitoring.
        """
        x, y = batch

        # Forward pass to get logits and compute loss and accuracy
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()

        # Logging training loss and accuracy for monitoring
        self.log('train_loss', loss, prog_bar=True, on_epoch=True)
        self.log('train_acc', acc, prog_bar=True, on_epoch=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        """
        Perform a validation step by computing the loss and accuracy for the current batch.
        Metrics logged for monitoring.
        """
        x, y = batch

        # Forward pass to get logits and compute loss and accuracy
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()

    # Logging validation loss and accuracy for monitoring
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_acc', acc, prog_bar=True, on_epoch=True)

    def test_step(self, batch, batch_idx):
        """ 
        Perform a test step by computing the loss and accuracy for the current batch. 
        Metrics logged for monitoring. 
        """
        x, y = batch

        # Forward pass to get logits and compute loss and accuracy
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()

        # Logging test loss and accuracy for monitoring
        self.log('test_loss', loss, prog_bar=True, on_epoch=True)
        self.log('test_acc', acc, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):
        """
        Configure the optimizer with different learning rates for the backbone and head of the model.
        """
        backbone_params = [param for param in self.model.features.parameters() if param.requires_grad]
        head_params = [param for param in self.model.classifier.parameters() if param.requires_grad]

        optimizer = torch.optim.AdamW([
            {'params': backbone_params, 'lr': self.hparams.lr_backbone},
            {'params': head_params, 'lr': self.hparams.lr_head}
        ], weight_decay=self.hparams.weight_decay)
        return optimizer

In [ ]:
pl.seed_everything(42, workers=True)

# Initialize the data module with the appropriate directories for training, validation, and testing datasets
data_module = PlantDataModule(
    train_dir='plant_img_train',
    val_dir='plant_img_val',
    test_dir='plant_img_test'
)

# Initialize the model with the number of classes equal to the number of source directories (classes) in the dataset
model = EfficientNetFineTuner(n_classes=len(source_dirs))

# Set up callbacks for model checkpointing and early stopping based on validation accuracy during both phases of training
checkpoint_callback_1 = ModelCheckpoint(
    monitor='val_acc',
    mode='max',
    save_top_k=1,
    filename='efficientnet-phase1-finetuned-{epoch:02d}-{val_acc:.4f}'
)

early_stopping_callback_1 = EarlyStopping(
    monitor='val_acc',
    mode='max',
    patience=5,
)

checkpoint_callback_2 = ModelCheckpoint(
    monitor='val_acc',
    mode='max',
    save_top_k=1,
    filename='efficientnet-phase2-finetuned-{epoch:02d}-{val_acc:.4f}'
)

early_stopping_callback_2 = EarlyStopping(
    monitor='val_acc',
    mode='max',
    patience=5,
)

Seed set to 42


### Model Training

In [ ]:
model.freeze_backbone()

# Set up the trainer for the first phase of training with the appropriate callbacks and settings for performance
trainer_1 = pl.Trainer(
    max_epochs=5,
    accelerator='auto',
    devices='auto',
    precision='32',
    callbacks=[checkpoint_callback_1, early_stopping_callback_1],
    log_every_n_steps=10,
)

trainer_1.fit(model, datamodule=data_module)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | model     | EfficientNet     | 4.0 M  | train | 0    
1 | criterion | CrossEntropyLoss | 0      | train | 0    
---------------------------------------------------------------
7.7 K     Trainable params
4.0 M     Non-trainable params
4.0 M     Total params
16.061    Total estimated model params size (MB)
338       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\missc\anaconda3\envs\dlenv\lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.


In [ ]:
model.unfreeze_last_n_blocks(n=3)

# Set up the trainer for the second phase of training with the appropriate callbacks and settings for performance
trainer_2 = pl.Trainer(
    max_epochs=20,
    accelerator='auto',
    devices='auto',
    precision='32',
    callbacks=[checkpoint_callback_2, early_stopping_callback_2],
    log_every_n_steps=10,
)

trainer_2.fit(model, datamodule=data_module)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | model     | EfficientNet     | 4.0 M  | train | 0    
1 | criterion | CrossEntropyLoss | 0      | train | 0    
---------------------------------------------------------------
3.2 M     Trainable params
851 K     Non-trainable params
4.0 M     Total params
16.061    Total estimated model params size (MB)
338       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

### Model Testing

In [8]:
trainer_2.test(model, datamodule=data_module, ckpt_path='best')

Restoring states from the checkpoint path at c:\Users\missc\EEE_ES\lightning_logs\version_10\checkpoints\efficientnet-phase2-finetuned-epoch=00-val_acc=0.9778.ckpt
Loaded model weights from the checkpoint at c:\Users\missc\EEE_ES\lightning_logs\version_10\checkpoints\efficientnet-phase2-finetuned-epoch=00-val_acc=0.9778.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9777777791023254
        test_loss           0.1611717790365219
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.1611717790365219, 'test_acc': 0.9777777791023254}]

### Inference

In [ ]:
@torch.inference_mode()
def classify_image(model, image_path, class_names, device="cpu", topk=3):
    """
    Classifies an input image using the trained model and returns the predicted label, confidence, and top-k predictions.
    """
    # Load the default weights for EfficientNet_B0 to get the correct normalization values for preprocessing the input image
    weights = EfficientNet_B0_Weights.DEFAULT
    
    # Define the preprocessing transforms to resize, center crop, convert to tensor, and normalize the input image using the same mean and std as used during training
    preprocess = transforms.Compose([
        transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=weights.transforms().mean, std=weights.transforms().std),
    ])

    # Load the input image, apply the preprocessing transforms, and add a batch dimension before moving it to the specified device for inference
    image = Image.open(image_path).convert("RGB")
    x = preprocess(image).unsqueeze(0).to(device)

    # Set the model to evaluation mode and move it to the specified device for inference
    model.eval()
    model.to(device)

    # Forward pass through the model to get logits, and apply softmax to get probabilities
    logits = model(x)
    probs = torch.softmax(logits, dim=1).squeeze(0)

    # Extract the top-k predictions along with their confidence scores
    confs, indices = torch.topk(probs, topk)
    topk_preds = [
        (class_names[int(idx)], float(conf))
        for idx, conf in zip(indices, confs)
    ]
    pred_label, pred_conf = topk_preds[0]

    return pred_label, pred_conf, topk_preds

In [ ]:
# Load the best checkpoint from the second phase of training
best_ckpt = trainer_2.checkpoint_callback.best_model_path
class_names = data_module.train_dataset.classes

model = EfficientNetFineTuner.load_from_checkpoint(
    best_ckpt,
    n_classes=len(class_names)
)

In [ ]:
image_path = 'sample_plant.png'
device = "cuda" if torch.cuda.is_available() else "cpu"

# Classify the input image and print the predicted label, confidence, and top-k predictions
pred_label, pred_conf, topk_preds = classify_image(model, image_path, class_names, device=device, topk=3)

print(f"Predicted label: {pred_label} (confidence: {pred_conf:.4f})")
print("Top-3 predictions:")
for label, conf in topk_preds:
    print(f"  {label}: {conf:.4f}")

Predicted label: haworthia_pumila (confidence: 0.9215)
Top-3 predictions:
  haworthia_pumila: 0.9215
  fittonia_albivenis: 0.0443
  echeveria_lilicana: 0.0108


### Recommendation

In [30]:
plant_care_df = pd.read_excel('plant_care_database.xlsx')

In [31]:
care_advice = plant_care_df[plant_care_df['Plant Name'] == pred_label]

In [32]:
care_advice

,Plant Name,Light Requirement,Moisture Requirement
0,haworthia_pumila,Medium indirect,Low


### Save Model

In [ ]:
# save_dir = "saved_models"
# os.makedirs(save_dir, exist_ok=True)

# final_ckpt = os.path.join(save_dir, "efficientnet_best2.ckpt")
# shutil.copy(best_ckpt, final_ckpt)

# print("Saved best model to:", final_ckpt)

Saved best model to: saved_models\efficientnet_best2.ckpt
